# 05_人类审批

**来源:** [LangChain Docs — Human-in-the-loop](https://docs.langchain.com/oss/python/deepagents/human-in-the-loop)

本 Notebook 演示 Deep Agents 的人类审批（Human-in-the-loop）工作流。某些工具操作可能较敏感，需要在执行前获得人工批准。Deep Agents 通过 LangGraph 的中断（interrupt）能力支持此功能。

## 1. 环境准备与导入

In [ ]:
# pip install deepagents langchain langgraph

from deepagents import create_deep_agent, CompiledSubAgent
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt
from langchain_core.utils.uuid import uuid7
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic

## 2. 基本配置

`interrupt_on` 参数接受一个字典，将工具名称映射到中断配置。每个工具可配置为：

- **`True`**：启用默认行为的中断（允许 approve, edit, reject, respond）
- **`False`**：禁用此工具的中断
- **`{"allowed_decisions": [...]}`**：自定义配置，指定允许的决策

In [ ]:
# 定义示例工具
@tool
def remove_file(path: str) -> str:
    """从文件系统中删除文件。"""
    return f"已删除 {path}"

@tool
def fetch_file(path: str) -> str:
    """从文件系统中读取文件。"""
    return f"{path} 的内容"

@tool
def notify_email(to: str, subject: str, body: str) -> str:
    """发送邮件。"""
    return f"已发送邮件给 {to}"

# Checkpointer 是 HITL 必需的
checkpointer = MemorySaver()

# 配置不同工具的中断行为
agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    tools=[remove_file, fetch_file, notify_email],
    interrupt_on={
        "remove_file": True,  # 默认：approve, edit, reject, respond
        "fetch_file": False,  # 不需要中断
        "notify_email": {"allowed_decisions": ["approve", "reject"]},  # 不允许编辑
    },
    checkpointer=checkpointer,  # 必填！
)

print("人类审批配置完成！")

## 3. 决策类型

`allowed_decisions` 列表控制人类审查工具调用时可执行的操作：

| 决策 | 描述 |
|------|------|
| `"approve"` | 按 Agent 提议的原始参数执行工具 |
| `"edit"` | 在执行前修改工具参数 |
| `"reject"` | 完全跳过此工具调用 |
| `"respond"` | 直接将人类消息作为工具结果返回，跳过执行——适用于"询问用户"风格的工具 |

可根据风险级别为每个工具定制：

```python
interrupt_on = {
    # 高风险：完全控制（approve, edit, reject）
    "delete_file": {"allowed_decisions": ["approve", "edit", "reject"]},
    "send_email": {"allowed_decisions": ["approve", "edit", "reject"]},

    # 中等风险：不允许编辑
    "write_file": {"allowed_decisions": ["approve", "reject"]},

    # 低风险：不中断
    "read_file": False,
    "list_files": False,
}
```

## 4. 处理中断

当中断触发时，Agent 暂停执行并返回控制。检查结果中的中断并相应处理。

In [ ]:
# 创建带线程 ID 的配置，用于状态持久化
config = {"configurable": {"thread_id": str(uuid7())}}

# 调用 Agent
result = agent.invoke(
    {"messages": [{"role": "user", "content": "删除文件 temp.txt"}]},
    config=config,
    version="v2",
)

# 检查是否被中断
if result.interrupts:
    # 提取中断信息
    interrupt_value = result.interrupts[0].value
    action_requests = interrupt_value["action_requests"]
    review_configs = interrupt_value["review_configs"]

    # 创建工具名称到审查配置的映射
    config_map = {cfg["action_name"]: cfg for cfg in review_configs}

    # 显示待处理的操作
    for action in action_requests:
        review_config = config_map[action["name"]]
        print(f"工具: {action['name']}")
        print(f"参数: {action['args']}")
        print(f"允许的决策: {review_config['allowed_decisions']}")

    # 模拟用户决策（批准删除）
    decisions = [
        {"type": "approve"}  # 用户批准了删除
    ]

    # 使用决策恢复执行
    result = agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config,  # 必须使用相同的配置！
        version="v2",
    )

# 处理最终结果
print("最终结果:", result.value["messages"][-1].content)

## 5. 多个工具调用

当 Agent 调用多个需要审批的工具时，所有中断会在一个中断中批量处理。必须按顺序为每个工具提供决策。

In [ ]:
# 多个工具调用的场景
config = {"configurable": {"thread_id": str(uuid7())}}

result = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "删除 temp.txt 并发送邮件给 admin@example.com"
    }]},
    config=config,
    version="v2",
)

if result.interrupts:
    interrupt_value = result.interrupts[0].value
    action_requests = interrupt_value["action_requests"]

    print(f"共有 {len(action_requests)} 个工具需要审批")

    # 按 action_requests 的顺序提供决策
    decisions = [
        {"type": "approve"},  # 第一个工具：delete_file - 批准
        {"type": "reject"}    # 第二个工具：send_email - 拒绝
    ]

    result = agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config,
        version="v2",
    )

print("最终结果:", result.value["messages"][-1].content)

## 6. 编辑工具参数

当 `"edit"` 在允许决策中时，可以在执行前修改工具参数。

In [ ]:
# 编辑工具参数示例
config = {"configurable": {"thread_id": str(uuid7())}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "发送通知邮件给所有人"}]},
    config=config,
    version="v2",
)

if result.interrupts:
    interrupt_value = result.interrupts[0].value
    action_request = interrupt_value["action_requests"][0]

    # Agent 提议的原始参数
    print(f"原始参数: {action_request['args']}")

    # 用户决定编辑收件人
    decisions = [{
        "type": "edit",
        "edited_action": {
            "name": action_request["name"],  # 必须包含工具名称
            "args": {
                "to": "team@company.com",
                "subject": "团队通知",
                "body": "请查看最新更新"
            }
        }
    }]

    result = agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config,
        version="v2",
    )

print("最终结果:", result.value["messages"][-1].content)

## 7. 子 Agent 中断

### 在工具调用上中断

每个子 Agent 可以有自己独立的 `interrupt_on` 配置，覆盖主 Agent 的设置。

In [ ]:
# 子 Agent 级别的中断配置
agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    tools=[remove_file, fetch_file],
    interrupt_on={
        "remove_file": True,
        "fetch_file": False,
    },
    subagents=[{
        "name": "file-manager",
        "description": "管理文件操作",
        "system_prompt": "你是一个文件管理助手。",
        "tools": [remove_file, fetch_file],
        "interrupt_on": {
            # 覆盖：在此子 Agent 中读取文件也需审批
            "remove_file": True,
            "fetch_file": True,  # 与主 Agent 不同！
        }
    }],
    checkpointer=checkpointer,
)

print("子 Agent 中断配置完成！")

### 在工具调用内部中断

子 Agent 工具可以直接调用 `interrupt()` 暂停执行等待审批。

In [ ]:
# 在工具内部使用 interrupt()
@tool(description="在执行操作前请求人工审批。")
def request_approval(action_description: str) -> str:
    """使用 interrupt() 原语请求人工审批。"""
    # interrupt() 暂停执行，返回传给 Command(resume=...) 的值
    approval = interrupt({
        "type": "approval_request",
        "action": action_description,
        "message": f"请批准或拒绝：{action_description}",
    })

    if approval.get("approved"):
        return f"操作 '{action_description}' 已批准。继续执行..."
    else:
        return f"操作 '{action_description}' 已拒绝。原因：{approval.get('reason', '未提供原因')}"


def main():
    checkpointer = MemorySaver()

    # 创建编译后的子 Agent
    compiled_subagent = create_agent(
        model="google_genai:gemini-3.5-flash",
        tools=[request_approval],
        name="approval-agent",
    )

    # 父 Agent
    parent_agent = create_deep_agent(
        model="google_genai:gemini-3.5-flash",
        checkpointer=checkpointer,
        subagents=[
            CompiledSubAgent(
                name="approval-agent",
                description="可以请求审批的 Agent",
                runnable=compiled_subagent,
            )
        ],
    )

    thread_id = "test_interrupt_directly"
    config = {"configurable": {"thread_id": thread_id}}

    print("调用 Agent - 子 Agent 将使用 request_approval 工具...")

    result = parent_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content="使用 task 工具启动 approval-agent 子 Agent。"
                    "让它使用 request_approval 工具请求批准'部署到生产环境'。"
                )
            ]
        },
        config=config,
        version="v2",
    )

    # 检查中断
    if result.interrupts:
        interrupt_value = result.interrupts[0].value
        print(f"\n中断收到！")
        print(f"  类型: {interrupt_value.get('type')}")
        print(f"  操作: {interrupt_value.get('action')}")
        print(f"  消息: {interrupt_value.get('message')}")

        print("\n使用 Command(resume={'approved': True}) 恢复...")
        result2 = parent_agent.invoke(
            Command(resume={"approved": True}),
            config=config,
            version="v2",
        )

        if not result2.interrupts:
            print("\n执行完成！")
            tool_msgs = [m for m in result2.value.get("messages", []) if m.type == "tool"]
            if tool_msgs:
                print(f"  工具结果: {tool_msgs[-1].content}")
        else:
            print("\n另一个中断发生")
    else:
        print("\n没有中断 - 模型可能未调用 request_approval")


if __name__ == "__main__":
    main()

## 8. 关键注意事项

### 始终使用 Checkpointer

```python
from langgraph.checkpoint.memory import MemorySaver
checkpointer = MemorySaver()
agent = create_deep_agent(..., checkpointer=checkpointer)  # HITL 必需
```

### 使用相同的线程 ID

恢复时必须使用相同的 `config` 和 `thread_id`：

```python
config = {"configurable": {"thread_id": "my-thread"}}
result = agent.invoke(input, config=config, version="v2")
# 恢复（使用相同 config）
result = agent.invoke(Command(resume={...}), config=config, version="v2")
```

### 决策顺序匹配操作

`decisions` 列表必须匹配 `action_requests` 的顺序：

```python
if result.interrupts:
    interrupt_value = result.interrupts[0].value
    action_requests = interrupt_value["action_requests"]

    decisions = []
    for action in action_requests:
        decision = get_user_decision(action)
        decisions.append(decision)

    result = agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config,
        version="v2",
    )
```

---

**参考链接**
- [Deep Agents — Human-in-the-loop](https://docs.langchain.com/oss/python/deepagents/human-in-the-loop)
- [LangGraph 中断文档](https://langchain-ai.github.io/langgraph/)
- [子 Agent](https://docs.langchain.com/oss/python/deepagents/subagents)
- [Checkpointer 文档](https://langchain-ai.github.io/langgraph/checkpoint/)